# UIT-VSFC Data Exploration

Exploratory checks for the Vietnamese student feedback sentiment dataset.

In [ ]:
from pathlib import Path
import sys

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.load import label_distribution, load_uit_vsfc

sns.set_theme(style="whitegrid")
LABEL_NAMES = {0: "negative", 1: "neutral", 2: "positive"}

In [ ]:
raw_dir = PROJECT_ROOT / "data" / "raw"
raw_paths = {
    "train": raw_dir / "train.csv",
    "dev": raw_dir / "dev.csv",
    "test": raw_dir / "test.csv",
}

if all(path.exists() for path in raw_paths.values()):
    train_df = pd.read_csv(raw_paths["train"])
    dev_df = pd.read_csv(raw_paths["dev"])
    test_df = pd.read_csv(raw_paths["test"])
else:
    train_df, dev_df, test_df = load_uit_vsfc()

splits = {"train": train_df, "dev": dev_df, "test": test_df}
print(pd.DataFrame({name: [len(frame)] for name, frame in splits.items()}, index=["rows"]).to_string())

In [ ]:
length_frames = []
summary_rows = []

for split_name, frame in splits.items():
    lengths = frame["sentence"].astype(str).str.split().str.len()
    length_frames.append(pd.DataFrame({"split": split_name, "word_len": lengths}))
    summary_rows.append(
        {
            "split": split_name,
            "mean": lengths.mean(),
            "median": lengths.median(),
            "p95": lengths.quantile(0.95),
            "min": lengths.min(),
            "max": lengths.max(),
        }
    )

length_df = pd.concat(length_frames, ignore_index=True)
length_summary = pd.DataFrame(summary_rows).round(2)
print(length_summary.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(
    data=length_df,
    x="word_len",
    hue="split",
    element="step",
    stat="density",
    common_norm=False,
    bins=40,
    ax=ax,
)
ax.set_title("Sentence length distribution")
ax.set_xlabel("Words per sentence")
ax.set_ylabel("Density")
plt.show()

In [ ]:
distribution_frames = []
for split_name, frame in splits.items():
    dist = label_distribution(frame).reset_index().rename(columns={"sentiment": "label"})
    dist["split"] = split_name
    dist["label_name"] = dist["label"].map(LABEL_NAMES)
    distribution_frames.append(dist)

label_dist_df = pd.concat(distribution_frames, ignore_index=True)
print(label_dist_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=label_dist_df, x="label_name", y="count", hue="split", ax=ax)
ax.set_title("Label distribution per split")
ax.set_xlabel("Sentiment")
ax.set_ylabel("Count")
plt.show()

In [ ]:
sample_rows = []
for label in sorted(train_df["sentiment"].unique()):
    samples = train_df.loc[train_df["sentiment"] == label, "sentence"].sample(
        n=min(5, (train_df["sentiment"] == label).sum()),
        random_state=42,
    )
    for sentence in samples:
        sample_rows.append(
            {"label": label, "label_name": LABEL_NAMES.get(label, str(label)), "sentence": sentence}
        )

print(pd.DataFrame(sample_rows).to_string(index=False))

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base", use_fast=False)
texts = train_df["sentence"].astype(str).tolist()

token_ids_seen = set()
unk_count = 0
token_count = 0
lengths_after_tokenization = []

for text in texts:
    token_ids = tokenizer.encode(text, add_special_tokens=True, truncation=True, max_length=128)
    content_ids = [
        token_id
        for token_id in token_ids
        if token_id not in {tokenizer.cls_token_id, tokenizer.sep_token_id, tokenizer.pad_token_id}
    ]
    token_ids_seen.update(content_ids)
    unk_count += sum(token_id == tokenizer.unk_token_id for token_id in content_ids)
    token_count += len(content_ids)
    lengths_after_tokenization.append(len(content_ids))

token_stats = pd.DataFrame(
    [
        {
            "tokenizer_vocab_size": tokenizer.vocab_size,
            "observed_token_types": len(token_ids_seen),
            "total_tokens": token_count,
            "unk_tokens": unk_count,
            "unk_rate_pct": round(100 * unk_count / max(token_count, 1), 4),
            "mean_tokens": round(float(np.mean(lengths_after_tokenization)), 2),
            "p95_tokens": round(float(np.quantile(lengths_after_tokenization, 0.95)), 2),
        }
    ]
)
print(token_stats.to_string(index=False))